# Tutorial 3: Squidpy Neighborhood Analysis

**Systems Biology — Spatial Proteomics Module**

## Learning Objectives
1. Build spatial neighbourhood graphs from cell coordinates
2. Compute **neighbourhood enrichment scores** — statistical tests for cell-cell co-localisation
3. Interpret z-score heatmaps in biological terms
4. Compute **co-occurrence scores** at multiple distance scales
5. Aggregate spatial patterns across all 132 images to identify robust interactions

## Biological background: why do spatial neighbours matter?

Cells communicate through direct contact and short-range secreted signals.  
The **neighbourhood enrichment test** asks: *does cell type A appear next to cell type B more often than expected by chance?*

- **Positive enrichment (high z-score)**: co-localisation — these cell types actively cluster together
- **Negative enrichment (low z-score)**: spatial avoidance — these cell types are found in different tissue regions
- **Near zero**: random mixing — no spatial preference

Examples in tumour immunology:
- Tumour cells cluster with each other (solid tumour mass)
- CD8 and CD4 T cells co-localise (T cell niches in immune stroma)
- Macrophages (MacCD163) often surround tumour — tumour-associated macrophages (TAMs)
- B cells and plasma cells cluster (tertiary lymphoid structures)

---

In [ ]:
import anndata as ad
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import squidpy as sq
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 100
sns.set_style('white')

print(f"Squidpy version: {sq.__version__}")
DATA_PATH = 'data/train_adata.h5ad'

---
## Part 1: Setup

Load data and create spatial coordinates. **Critical**: image IDs are strings (filenames), not integers.

In [ ]:
adata = ad.read_h5ad(DATA_PATH)

# Create arcsinh expression layer
if 'exprs_arcsinh' not in adata.layers:
    adata.layers['exprs_arcsinh'] = np.arcsinh(adata.layers['exprs'] / 5)

# Squidpy requires coordinates in .obsm['spatial']
adata.obsm['spatial'] = adata.obs[['Pos_X', 'Pos_Y']].values

print(f"Loaded {adata.n_obs:,} cells from {adata.obs['image'].nunique()} images")
print(f"Cell types: {sorted(adata.obs['celltypes'].unique().tolist())}")

# Image IDs are strings (filenames), not integers!
all_images = adata.obs['image'].unique()
print(f"\nExample image ID: {all_images[0]}")

In [ ]:
# Select a large image for detailed analysis
img_sizes = adata.obs.groupby('image', observed=True).size()
IMAGE_ID = img_sizes.idxmax()
adata_single = adata[adata.obs['image'] == IMAGE_ID].copy()

print(f"Selected: {IMAGE_ID}")
print(f"Cells: {adata_single.n_obs}")
print(f"Indication: {adata_single.obs['Indication'].iloc[0]}")
print(f"\nCell type counts:")
print(adata_single.obs['celltypes'].value_counts().to_string())

---
## Part 2: Building the Spatial Neighbourhood Graph

Squidpy's `sq.gr.spatial_neighbors` connects cells within a given radius.  
Result stored in `adata.obsp['spatial_connectivities']` (sparse matrix).

**Radius choice:** 20 μm ≈ 1–2 cell diameters in IMC (cells ~10–15 μm diameter).  
This captures direct cell-cell contacts and very short-range communication.

In [ ]:
RADIUS = 20  # micrometers

sq.gr.spatial_neighbors(
    adata_single,
    coord_type='generic',
    radius=RADIUS,
    spatial_key='spatial'
)

conn = adata_single.obsp['spatial_connectivities']
print(f"Spatial graph built (radius = {RADIUS} um):")
print(f"  Nodes (cells) : {conn.shape[0]}")
print(f"  Edges (pairs) : {conn.nnz // 2}  (undirected)")
print(f"  Mean degree   : {conn.nnz / conn.shape[0]:.1f} neighbours per cell")
print(f"  Max degree    : {np.array(conn.sum(axis=1)).max():.0f}")

In [ ]:
# Degree distribution: how many neighbours does each cell have?
degrees = np.array(conn.sum(axis=1)).flatten()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(degrees, bins=40, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].set_xlabel(f'Number of neighbours (within {RADIUS} um)')
axes[0].set_ylabel('Number of cells')
axes[0].set_title('Cell neighbourhood degree distribution')
axes[0].axvline(degrees.mean(), color='red', linestyle='--', label=f'Mean = {degrees.mean():.1f}')
axes[0].legend()

# Degree by cell type (which cell types are more densely packed?)
degree_df = pd.DataFrame({'degree': degrees, 'celltype': adata_single.obs['celltypes'].values})
order = degree_df.groupby('celltype')['degree'].median().sort_values(ascending=False).index
sns.boxplot(data=degree_df, x='degree', y='celltype', order=order, ax=axes[1],
            palette='husl', orient='h')
axes[1].set_xlabel(f'Neighbours within {RADIUS} um')
axes[1].set_title('Neighbourhood size by cell type\n(reflects local packing density)')

plt.suptitle('Spatial graph properties', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Visualise the graph on a spatial scatter plot (subsample for clarity)
# Show edges for a small region of the image

coords = adata_single.obsm['spatial']
celltypes = adata_single.obs['celltypes'].values

# Focus on a 150x150 um region (top-left corner)
region_mask = (coords[:, 0] < 150) & (coords[:, 1] < 150)
region_idx = np.where(region_mask)[0]

cts = sorted(adata_single.obs['celltypes'].unique())
palette = dict(zip(cts, sns.color_palette('tab20', len(cts))))

fig, ax = plt.subplots(figsize=(10, 10))

# Draw edges for the region
conn_dense = conn.toarray()
for i in region_idx:
    for j in region_idx:
        if i < j and conn_dense[i, j] > 0:
            ax.plot([coords[i, 0], coords[j, 0]],
                    [coords[i, 1], coords[j, 1]],
                    color='lightgrey', linewidth=0.5, alpha=0.6, zorder=1)

# Draw cells coloured by type
for ct in cts:
    m = region_mask & (celltypes == ct)
    if m.sum() > 0:
        ax.scatter(coords[m, 0], coords[m, 1], s=60, color=palette[ct],
                   label=ct, edgecolors='white', linewidth=0.5, zorder=2)

ax.set_aspect('equal')
ax.invert_yaxis()
ax.legend(loc='upper left', bbox_to_anchor=(1, 1), fontsize=8, markerscale=1, title='Cell type')
ax.set_title(f'Spatial neighbourhood graph (first 150x150 um)\nEdges = pairs within {RADIUS} um radius',
             fontsize=11)
ax.set_xlabel('X (um)')
ax.set_ylabel('Y (um)')
plt.tight_layout()
plt.show()

---
## Part 3: Neighbourhood Enrichment Analysis

### How the test works

1. Count observed neighbour pairs for each cell type combination (A, B)
2. **Permute cell type labels** N times and recount pairs under random assignment
3. Z-score = (observed - mean_random) / std_random

**Interpretation:**
- z >> 0: A and B are neighbours more than expected → spatial co-localisation
- z << 0: A and B avoid each other spatially
- z ≈ 0: random mixing

In [ ]:
# Compute neighbourhood enrichment
# n_perms=200 is fast (~seconds) and sufficient for exploration
# Use n_perms=1000 for publication-quality results
sq.gr.nhood_enrichment(
    adata_single,
    cluster_key='celltypes',
    n_perms=200,
    seed=42
)

zscore = adata_single.uns['celltypes_nhood_enrichment']['zscore']
print(f"Z-score matrix shape: {zscore.shape}")
print(f"Z-score range: [{zscore.min():.2f}, {zscore.max():.2f}]")
print(f"\nStrongly enriched pairs (z > 5):")

cell_type_names = adata_single.obs['celltypes'].cat.categories.tolist()
for i in range(len(cell_type_names)):
    for j in range(i, len(cell_type_names)):
        if zscore[i, j] > 5:
            print(f"  {cell_type_names[i]} <-> {cell_type_names[j]}: z = {zscore[i, j]:.1f}")

In [ ]:
# Official Squidpy visualisation
sq.pl.nhood_enrichment(
    adata_single,
    cluster_key='celltypes',
    figsize=(10, 8),
    title='Neighbourhood Enrichment Z-scores (single image)'
)
plt.show()

In [ ]:
# Custom heatmap with cleaner layout and biological annotation
zscore_df = pd.DataFrame(
    zscore,
    index=cell_type_names,
    columns=cell_type_names
)

# Symmetric matrix -- keep upper triangle, mirror to lower for display
fig, ax = plt.subplots(figsize=(11, 9))
mask = np.zeros_like(zscore, dtype=bool)
# No mask - show full matrix
sns.heatmap(
    zscore_df,
    cmap='RdBu_r',
    center=0,
    vmin=-10, vmax=10,
    square=True,
    linewidths=0.5,
    linecolor='white',
    ax=ax,
    cbar_kws={'label': 'Neighbourhood enrichment (z-score)',
              'shrink': 0.7}
)
ax.set_title('Cell-cell neighbourhood enrichment\n'
             'Red = co-localised | Blue = spatially avoided | White = random',
             fontsize=12)
ax.tick_params(axis='x', rotation=45)
ax.tick_params(axis='y', rotation=0)
plt.tight_layout()
plt.show()

# Print top enriched pairs (excluding self-enrichment)
zscore_nodiag = zscore_df.copy()
np.fill_diagonal(zscore_nodiag.values, np.nan)

print("Top 10 ENRICHED cell type pairs (highest z-score):")
flat = zscore_nodiag.stack().sort_values(ascending=False)
for (ct1, ct2), z in flat.head(10).items():
    print(f"  {ct1:15s} <-> {ct2:15s}  z = {z:.2f}")

print("\nTop 10 AVOIDED cell type pairs (lowest z-score):")
for (ct1, ct2), z in flat.tail(10).items():
    print(f"  {ct1:15s} <-> {ct2:15s}  z = {z:.2f}")

### Exercise 3.1: Interpret the Enrichment Matrix

**Question 1:** Which cell type pair has the **strongest self-enrichment** (highest diagonal value)? Why does this make biological sense?

**Question 2:** Is there enrichment between **B cells** and **plasma cells**? What does this suggest about lymphoid structure in the tissue?

**Question 3:** Is **Tumor-CD8** enrichment positive or negative? What does this mean for the anti-tumour immune response in this image?

In [ ]:
# Your analysis here
print("Self-enrichment (diagonal):")
diag_vals = pd.Series(
    np.diag(zscore),
    index=cell_type_names
).sort_values(ascending=False)
print(diag_vals.round(2).to_string())

print(f"\nB <-> plasma enrichment: z = {zscore_df.loc['B', 'plasma']:.2f}")
print(f"Tumor <-> CD8 enrichment: z = {zscore_df.loc['Tumor', 'CD8']:.2f}")
print(f"Tumor <-> MacCD163 enrichment: z = {zscore_df.loc['Tumor', 'MacCD163']:.2f}")

---
## Part 4: Co-occurrence Score at Multiple Distances

The neighbourhood enrichment test uses a single radius cutoff.  
The **co-occurrence score** tests enrichment at *many distance scales simultaneously*, revealing:
- At what distance does enrichment peak?
- Does it persist at long range?

`sq.gr.co_occurrence` computes: for each cell of type A, how much more likely is type B within annuli at increasing distances, compared to random?

In [ ]:
# Co-occurrence analysis (may take ~1-2 minutes)
sq.gr.co_occurrence(
    adata_single,
    cluster_key='celltypes',
    spatial_key='spatial',
    n_splits=5  # reduce for speed
)

occ   = adata_single.uns['celltypes_co_occurrence']['occ']       # (n_types, n_types, n_bins)
bins  = adata_single.uns['celltypes_co_occurrence']['interval']  # (n_bins+1,) bin edges
dist_centers = (np.array(bins[:-1]) + np.array(bins[1:])) / 2   # midpoints of each bin

print("Co-occurrence computed.")
print(f"occ shape   : {occ.shape}  (cell_types x cell_types x distance_bins)")
print(f"n_bins      : {len(dist_centers)}")
print(f"Max distance: {dist_centers[-1]:.1f} um")

In [ ]:
# Official Squidpy plot for key interactions
sq.pl.co_occurrence(
    adata_single,
    cluster_key='celltypes',
    clusters=['Tumor', 'CD8'],
    figsize=(12, 5)
)
plt.suptitle('Co-occurrence score: how enriched is each cell type\n'
             'within increasing distance of Tumor and CD8 cells', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# Custom multi-panel: focus on biologically interesting pairs
local_types = adata_single.obs['celltypes'].cat.categories.tolist()

pairs_to_show = [
    ('Tumor',    'CD8',       'Tumor-CD8 (immunotherapy axis)'),
    ('Tumor',    'MacCD163',  'Tumor-Macrophage (TAM axis)'),
    ('CD8',      'CD4',       'CD8-CD4 (T cell niches)'),
    ('B',        'plasma',    'B-plasma (lymphoid structures)'),
]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for ax, (ct1, ct2, title) in zip(axes, pairs_to_show):
    if ct1 not in local_types or ct2 not in local_types:
        ax.text(0.5, 0.5, f'{ct1} or {ct2} not in this image',
                transform=ax.transAxes, ha='center', va='center')
        ax.set_title(title, fontsize=10)
        continue

    idx1 = local_types.index(ct1)
    idx2 = local_types.index(ct2)

    # occ shape: (n_types, n_types, n_bins)
    # occ[i, j, :] = co-occurrence scores for cells of type j around cells of type i
    scores_12 = occ[idx1, idx2, :]   # ct2 around ct1
    scores_21 = occ[idx2, idx1, :]   # ct1 around ct2

    ax.plot(dist_centers, scores_12, 'b-o', markersize=3, linewidth=1.5,
            label=f'{ct2} near {ct1}')
    ax.plot(dist_centers, scores_21, 'r-o', markersize=3, linewidth=1.5,
            label=f'{ct1} near {ct2}')
    ax.axhline(y=1, color='grey', linestyle='--', alpha=0.7, label='random (=1)')
    ax.set_xlabel('Distance (um)')
    ax.set_ylabel('Co-occurrence score')
    ax.set_title(title, fontsize=10)
    ax.legend(fontsize=8)

plt.suptitle('Co-occurrence scores at multiple spatial scales\n'
             'Values > 1: enriched; < 1: depleted; = 1: random',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Part 5: Aggregating Across All 132 Images

Single-image analysis is noisy. To find **robust** patterns, we aggregate enrichment scores across all images.

**Challenge:** Not all images contain all 14 cell types. We handle this by:
1. Computing enrichment only for images with ≥50 cells
2. Using the full cell type list from all data so matrices are comparable
3. Reporting mean and standard deviation across images

In [ ]:
# All 14 cell types from the full dataset
ALL_CELLTYPES = sorted(adata.obs['celltypes'].unique().tolist())
N_CT = len(ALL_CELLTYPES)
ct_idx = {ct: i for i, ct in enumerate(ALL_CELLTYPES)}

print(f"Cell types ({N_CT}): {ALL_CELLTYPES}")

# NOTE: Processing all 132 images with 500 permutations each would take ~30 min.
# We cap at MAX_IMAGES for interactive use. Increase for a final analysis.
MAX_IMAGES = 60
images_to_process = all_images[:MAX_IMAGES]

print(f"\nProcessing {len(images_to_process)} of {len(all_images)} images...")
print("(Increase MAX_IMAGES to process all 132 images)")

# Accumulate z-score matrices
zscore_stack = []
image_meta   = []
skipped = 0

for img_id in images_to_process:
    sub = adata[adata.obs['image'] == img_id].copy()

    if sub.n_obs < 50 or sub.obs['celltypes'].nunique() < 2:
        skipped += 1
        continue

    # Build spatial graph
    sq.gr.spatial_neighbors(sub, coord_type='generic', radius=20, spatial_key='spatial')

    # Neighbourhood enrichment (100 perms is enough for exploration)
    try:
        sq.gr.nhood_enrichment(sub, cluster_key='celltypes', n_perms=100, seed=42)
    except Exception:
        skipped += 1
        continue

    # Map local cell types → global 14x14 matrix
    local_types  = sub.obs['celltypes'].cat.categories.tolist()
    local_zscore = sub.uns['celltypes_nhood_enrichment']['zscore']

    full_mat = np.full((N_CT, N_CT), np.nan)
    for li, lct in enumerate(local_types):
        if lct not in ct_idx:
            continue
        gi = ct_idx[lct]
        for lj, lct2 in enumerate(local_types):
            if lct2 not in ct_idx:
                continue
            gj = ct_idx[lct2]
            full_mat[gi, gj] = local_zscore[li, lj]

    zscore_stack.append(full_mat)
    image_meta.append({
        'image':      img_id,
        'indication': sub.obs['Indication'].iloc[0],
        'n_cells':    sub.n_obs,
        'tumor_cd8_zscore': full_mat[ct_idx['Tumor'], ct_idx['CD8']]
            if ('Tumor' in ct_idx and 'CD8' in ct_idx) else np.nan
    })

zscore_array = np.array(zscore_stack)
meta_df = pd.DataFrame(image_meta)

print(f"\nProcessed: {len(zscore_stack)} images")
print(f"Skipped:   {skipped} images (too few cells or enrichment failed)")

In [ ]:
# Mean and std z-score across all processed images
zscore_mean = np.nanmean(zscore_array, axis=0)
zscore_std  = np.nanstd(zscore_array, axis=0)

mean_df = pd.DataFrame(zscore_mean, index=ALL_CELLTYPES, columns=ALL_CELLTYPES)
std_df  = pd.DataFrame(zscore_std,  index=ALL_CELLTYPES, columns=ALL_CELLTYPES)

fig, axes = plt.subplots(1, 2, figsize=(20, 8))

# Mean enrichment
sns.heatmap(
    mean_df, cmap='RdBu_r', center=0, vmin=-5, vmax=5,
    square=True, linewidths=0.5, linecolor='white', ax=axes[0],
    cbar_kws={'label': 'Mean z-score', 'shrink': 0.7}
)
axes[0].set_title(f'Mean neighbourhood enrichment\n(aggregated over {len(zscore_stack)} images)',
                  fontsize=11)
axes[0].tick_params(axis='x', rotation=45)

# Std (variability across images)
sns.heatmap(
    std_df, cmap='YlOrRd', vmin=0,
    square=True, linewidths=0.5, linecolor='white', ax=axes[1],
    cbar_kws={'label': 'Std of z-score', 'shrink': 0.7}
)
axes[1].set_title(f'Variability of neighbourhood enrichment\n(std across {len(zscore_stack)} images)',
                  fontsize=11)
axes[1].tick_params(axis='x', rotation=45)

plt.suptitle('Robust cell-cell spatial interactions across all images',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Rank cell type pairs by mean enrichment (upper triangle only)
pairs = []
for i, ct1 in enumerate(ALL_CELLTYPES):
    for j, ct2 in enumerate(ALL_CELLTYPES[i:], start=i):
        z_mean = zscore_mean[i, j]
        z_std  = zscore_std[i, j]
        n_obs  = (~np.isnan(zscore_array[:, i, j])).sum()
        pairs.append({'ct1': ct1, 'ct2': ct2, 'z_mean': z_mean,
                      'z_std': z_std, 'n_images': n_obs})

pairs_df = pd.DataFrame(pairs).dropna(subset=['z_mean'])

print("Top 15 ENRICHED cell type pairs (mean z-score, excluding self):")
top_enriched = pairs_df[pairs_df['ct1'] != pairs_df['ct2']].nlargest(15, 'z_mean')
for _, row in top_enriched.iterrows():
    print(f"  {row.ct1:15s} <-> {row.ct2:15s}  "
          f"z={row.z_mean:+6.2f} (+/-{row.z_std:.2f}), n={row.n_images:.0f} images")

print("\nTop 10 AVOIDED cell type pairs:")
top_avoided = pairs_df[pairs_df['ct1'] != pairs_df['ct2']].nsmallest(10, 'z_mean')
for _, row in top_avoided.iterrows():
    print(f"  {row.ct1:15s} <-> {row.ct2:15s}  "
          f"z={row.z_mean:+6.2f} (+/-{row.z_std:.2f}), n={row.n_images:.0f} images")

---
## Part 6: Image-Level Variability — Tumour-Immune Spatial Axis

The Tumour-CD8 enrichment score varies across images and cancer types.  
Images with high positive z-scores are likely immune-infiltrated (hot);  
images with strongly negative scores are immune-excluded (cold).

In [ ]:
# Tumor-CD8 z-score distribution across images and cancer types
valid = meta_df.dropna(subset=['tumor_cd8_zscore'])

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Distribution by indication
ind_order = valid.groupby('indication')['tumor_cd8_zscore'].median().sort_values(ascending=False).index
palette = sns.color_palette('Set2', len(ind_order))

sns.boxplot(data=valid, x='indication', y='tumor_cd8_zscore',
            order=ind_order, palette=palette, ax=axes[0])
sns.swarmplot(data=valid, x='indication', y='tumor_cd8_zscore',
              order=ind_order, color='black', size=3, alpha=0.5, ax=axes[0])
axes[0].axhline(y=0, color='grey', linestyle='--', alpha=0.7)
axes[0].set_xlabel('Cancer type')
axes[0].set_ylabel('Tumour-CD8 neighbourhood z-score')
axes[0].set_title('Tumour-CD8 spatial enrichment by cancer type\n'
                  '>0: CD8 cells preferentially near tumour')

# Histogram of z-scores across all images
axes[1].hist(valid['tumor_cd8_zscore'], bins=30, color='steelblue',
             edgecolor='white', alpha=0.8)
axes[1].axvline(0, color='grey', linestyle='--', alpha=0.7, label='random')
axes[1].axvline(valid['tumor_cd8_zscore'].mean(), color='red', linestyle='-',
                label=f'mean = {valid["tumor_cd8_zscore"].mean():.2f}')
axes[1].set_xlabel('Tumour-CD8 neighbourhood z-score')
axes[1].set_ylabel('Number of images')
axes[1].set_title('Distribution of Tumour-CD8 enrichment across all images')
axes[1].legend()

plt.tight_layout()
plt.show()

print("Fraction of images with positive Tumor-CD8 enrichment:")
print(f"  {(valid['tumor_cd8_zscore'] > 0).mean():.1%} of images")
print(f"  Mean z-score: {valid['tumor_cd8_zscore'].mean():.2f}")

In [ ]:
# Find images with extreme Tumour-CD8 enrichment
top_enriched_imgs = meta_df.nlargest(3, 'tumor_cd8_zscore')[['image','indication','n_cells','tumor_cd8_zscore']]
top_depleted_imgs = meta_df.nsmallest(3, 'tumor_cd8_zscore')[['image','indication','n_cells','tumor_cd8_zscore']]

print("Images with strongest Tumour-CD8 CO-LOCALISATION (z >> 0):")
for _, row in top_enriched_imgs.iterrows():
    print(f"  [{row.indication}] {row.image[-50:]:50s}  z={row.tumor_cd8_zscore:.2f}")

print("\nImages with strongest Tumour-CD8 AVOIDANCE (z << 0):")
for _, row in top_depleted_imgs.iterrows():
    print(f"  [{row.indication}] {row.image[-50:]:50s}  z={row.tumor_cd8_zscore:.2f}")

### Exercise 3.2: Compare Enrichment Patterns by Cancer Type

**Task:** Compute the mean neighbourhood enrichment separately for each cancer indication (BREAS, HN, THOR, GU, GI).  
Plot 5 heatmaps side by side.

**Question:** Which cancer type has the most distinct spatial organisation?  
Are there interactions that are consistently present across all types, or some that are indication-specific?

*Hint:* Use `meta_df['indication']` to group the `zscore_array` by cancer type.

In [ ]:
# Your code here
indications = sorted(meta_df['indication'].unique())
fig, axes = plt.subplots(1, len(indications), figsize=(5*len(indications), 7))

for ax, ind in zip(axes, indications):
    ind_mask = meta_df['indication'] == ind
    ind_zscores = zscore_array[ind_mask.values]
    ind_mean = np.nanmean(ind_zscores, axis=0)
    ind_df = pd.DataFrame(ind_mean, index=ALL_CELLTYPES, columns=ALL_CELLTYPES)

    sns.heatmap(
        ind_df, cmap='RdBu_r', center=0, vmin=-5, vmax=5,
        square=True, linewidths=0.3, linecolor='white',
        ax=ax, cbar=(ind == indications[-1]),
        cbar_kws={'label': 'Mean z-score', 'shrink': 0.7} if ind == indications[-1] else {}
    )
    ax.set_title(f'{ind}\n(n={ind_mask.sum()} images)', fontsize=10)
    ax.tick_params(axis='x', rotation=45, labelsize=7)
    ax.tick_params(axis='y', labelsize=7)
    if ind != indications[0]:
        ax.set_ylabel('')

plt.suptitle('Neighbourhood enrichment by cancer type', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Summary

| Concept | Key takeaway |
|---|---|
| `sq.gr.spatial_neighbors` | Builds graph; radius=20 um = 1-2 cell diameters for direct contact |
| `sq.gr.nhood_enrichment` | Permutation test: z-score of observed vs random neighbour frequencies |
| z >> 0 | Co-localisation — cells cluster together (e.g., Tumor self-enrichment) |
| z << 0 | Spatial avoidance — cell types occupy different tissue regions |
| Co-occurrence | Tests enrichment at multiple distance scales; reveals range of interactions |
| Multi-image aggregation | Required for robust conclusions; single images are noisy |

**Key biological findings:**
- Tumour cells strongly self-enrich (solid tumour masses)
- T cell subtypes (CD4, CD8, Treg) co-localise (immune stroma niches)
- B cells and plasma cells cluster (tertiary lymphoid structures)
- Tumour-CD8 enrichment varies by cancer type and patient — this is the spatial immune infiltration axis

**Interpretation principle:** A high Tumour-CD8 z-score means CD8 cells are physically next to tumour cells more than expected — they are in a position to kill. A low z-score means immune exclusion, often indicating immune evasion.

---